In [14]:
import duckdb
import os
from pathlib import Path

# ensuring that os.chdir is idempotent and that we are in the project root directory,
# not inside the notebooks directory
if 'notebooks' not in os.listdir(Path.cwd()):
    print("Still inside notebooks directory, changing to project root directory.")
    os.chdir(Path.cwd().parent)
    print("Current working directory after change: ", Path.cwd())
else:
    print(f"Already in parent directory (current working directory: {Path.cwd()})")


# load config
from src.io.load_config import load_config
sw_config = load_config()['space_weather']
omni_config = load_config()['omni']

Already in parent directory (current working directory: d:\data-sci-projects\space-weather-project-scrub)


# Test endpoints from CDAWeb HAPI

CDAWeb HAPI description can be found at https://cdaweb.gsfc.nasa.gov/hapi

In [15]:
import pandas as pd
import requests
from io import StringIO
import json

## `/info`

In [16]:
def fetch_info_omni_hro2_1min(dataset_id="OMNI_HRO2_1MIN",
                            base_url="https://cdaweb.gsfc.nasa.gov/hapi",
                            ):

    # as per HAPI description, endpoints respond to HTTP GET requests 
    response = requests.get(base_url + '/info',
                            params={"id": dataset_id},
                            timeout=120)
    
    response.raise_for_status()

    return response.json()

In [17]:
info_endpoint_test = fetch_info_omni_hro2_1min()

In [18]:
info_endpoint_test

{'HAPI': '2.0',
 'resourceURL': 'https://cdaweb.gsfc.nasa.gov/misc/NotesO.html#OMNI_HRO2_1MIN',
 'contact': 'J.H. King, N. Papatashvilli @ AdnetSystems, NASA GSFC',
 'parameters': [{'name': 'Time',
   'length': 24,
   'units': 'UTC',
   'type': 'isotime',
   'fill': None},
  {'name': 'IMF',
   'description': 'OMNI ID code for the source spacecraft for time-shifted IMF values (see OMNI documentation link for codes)',
   'units': None,
   'type': 'integer',
   'fill': '99'},
  {'name': 'PLS',
   'description': 'OMNI ID code for the source spacecraft  for time-shifted IP plasma values (see OMNI documentation link for codes)',
   'units': None,
   'type': 'integer',
   'fill': '99'},
  {'name': 'IMF_PTS',
   'description': 'Number of fine time scale points in IMF averages',
   'units': None,
   'type': 'integer',
   'fill': '999'},
  {'name': 'PLS_PTS',
   'description': 'Number of fine time scale points in plasma averages',
   'units': None,
   'type': 'integer',
   'fill': '999'},
  {'na

In [10]:
pd.DataFrame(info_endpoint_test['parameters'])

,name,length,units,type,fill,description
0,Time,24.0,UTC,isotime,None,NaN
1,IMF,NaN,None,integer,99,OMNI ID code for the source spacecraft for tim...
2,PLS,NaN,None,integer,99,OMNI ID code for the source spacecraft for ti...
3,IMF_PTS,NaN,None,integer,999,Number of fine time scale points in IMF averages
4,PLS_PTS,NaN,None,integer,999,Number of fine time scale points in plasma ave...
5,percent_interp,NaN,None,integer,999,Percent interpolated
6,Timeshift,NaN,seconds,integer,999999,Timeshift (seconds)
7,RMS_Timeshift,NaN,seconds,integer,999999,RMS Timeshift (seconds)
8,RMS_phase,NaN,nT,double,99.99,"RMS, Phase front normal (nT)"
9,Time_btwn_obs,NaN,seconds,integer,999999,Time between observations (seconds)


In [ ]:
query_var = 'BX_GSE'
display(
    pd.DataFrame(info_endpoint_test['parameters']).query(f"name=='{query_var}'")
)

display(
    pd.DataFrame(info_endpoint_test['parameters']).query(f"name=='{query_var}'").fill.item()
)


## `/data` (with params)

In [29]:
# https://cdaweb.gsfc.nasa.gov/hapi/data?id=OMNI_HRO2_1MIN&parameters=F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure&time.min=2021-11-21T00:00:00Z&time.max=2021-11-22T00:00:00Z&format=csv
def fetch_data_omni_hro2_1min(start_utc: str,
                         end_utc: str,
                         vars: str="F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure",
                         format=None
                         ):

    url = "https://cdaweb.gsfc.nasa.gov/hapi/data"

    params = {
        "id": "OMNI_HRO2_1MIN",
        "time.min": start_utc,
        "time.max": end_utc,
        "format": format
    }
    
    if vars:
        # no intermediate or trailing whitespaces allowed
        vars = vars.replace(" ", "")
        params['parameters'] = vars

    print("Requesting solar dataset from OMNI_HRO2_1MIN...")
    response = requests.get(url, params=params, timeout=120)
    print(response.text)
    response.raise_for_status()
    print("Request succeeded.")
    
    if not format or format == 'csv':
        df = pd.read_csv(StringIO(response.text),
                        comment="#",
                        header=None)
        df.columns = ["time"] + vars.split(',')
        return df
    

    if format == 'json':
        try:
            return response.json()
        except Exception as e:
            print("⚠️⚠️ MALFOMED JSON RESPONSE ⚠️⚠️. Returning raw response instead.")
            print(repr(e))
            return response.text


### Return csv

#### Success

In [7]:
fetch_data_omni_hro2_1min(
    start_utc="2021-11-21T00:00:00Z",
    end_utc="2021-11-22T00:00:00Z"
)

Requesting solar dataset from OMNI_HRO2_1MIN...
2021-11-21T00:00:00.000Z,4.79,1.87,-2.40,-3.37,99999.9,999.99,99.99
2021-11-21T00:01:00.000Z,5.73,4.25,0.29,2.60,99999.9,999.99,99.99
2021-11-21T00:02:00.000Z,4.90,1.72,-3.04,-3.39,583.6,4.70,3.20
2021-11-21T00:03:00.000Z,4.92,2.03,-3.04,-3.23,583.6,4.70,3.20
2021-11-21T00:04:00.000Z,5.00,1.68,-2.93,-3.67,583.6,4.70,3.20
2021-11-21T00:05:00.000Z,5.09,1.65,-3.44,-3.23,583.0,4.62,3.14
2021-11-21T00:06:00.000Z,5.13,1.61,-3.50,-3.36,583.0,4.62,3.14
2021-11-21T00:07:00.000Z,5.57,3.56,-0.86,0.89,574.7,4.18,2.76
2021-11-21T00:08:00.000Z,5.64,3.86,0.77,2.73,574.5,4.19,2.77
2021-11-21T00:09:00.000Z,5.35,4.26,-0.27,1.52,568.1,4.37,2.82
2021-11-21T00:10:00.000Z,5.46,4.09,-0.17,1.75,569.0,4.36,2.82
2021-11-21T00:11:00.000Z,5.36,3.95,-0.69,0.85,569.3,4.35,2.82
2021-11-21T00:12:00.000Z,5.34,4.09,-0.32,1.00,578.4,4.73,3.16
2021-11-21T00:13:00.000Z,5.20,3.99,-0.29,0.88,574.5,4.53,2.99
2021-11-21T00:14:00.000Z,5.08,3.89,-0.34,1.14,99999.9,999.99,99.99
202

,time,F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure
0,2021-11-21T00:00:00.000Z,4.79,1.87,-2.40,-3.37,99999.9,999.99,99.99
1,2021-11-21T00:01:00.000Z,5.73,4.25,0.29,2.60,99999.9,999.99,99.99
2,2021-11-21T00:02:00.000Z,4.90,1.72,-3.04,-3.39,583.6,4.70,3.20
3,2021-11-21T00:03:00.000Z,4.92,2.03,-3.04,-3.23,583.6,4.70,3.20
4,2021-11-21T00:04:00.000Z,5.00,1.68,-2.93,-3.67,583.6,4.70,3.20
...,...,...,...,...,...,...,...,...
1435,2021-11-21T23:55:00.000Z,3.69,2.28,-2.76,0.48,612.4,2.33,1.75
1436,2021-11-21T23:56:00.000Z,3.76,1.76,-3.15,0.31,619.3,2.45,1.88
1437,2021-11-21T23:57:00.000Z,3.75,-0.65,-3.07,-1.14,635.0,2.35,1.90
1438,2021-11-21T23:58:00.000Z,3.95,-1.41,-3.01,-2.11,643.1,2.31,1.91


#### Fail: no data in time range

In [51]:
fetch_data_omni_hro2_1min(
    start_utc="1500-11-21T00:00:00Z",
    end_utc="1501-11-22T00:00:00Z"
)

Requesting solar dataset from OMNI_HRO2_1MIN...
{
"HAPI": "2.0",
"status": {"code": 1201, "message": "OK - no data for time range"}
}

Request succeeded.


ParserError: Error tokenizing data. C error: Expected 1 fields in line 2, saw 2


In [73]:
# stopDate for omni_hro2_1min is 2026-04-13
fetch_data_omni_hro2_1min(
    start_utc="2026-07-01T00:00:00Z",
    end_utc="2026-07-02T00:00:00Z"
)

Requesting solar dataset from OMNI_HRO2_1MIN...
{
"HAPI": "2.0",
"status": {"code": 1201, "message": "OK - no data for time range"}
}

Request succeeded.


ParserError: Error tokenizing data. C error: Expected 1 fields in line 2, saw 2


#### Fail: incorrect variable name

In [52]:
fetch_data_omni_hro2_1min(
    start_utc="1500-11-21T00:00:00Z",
    end_utc="1501-11-22T00:00:00Z",
    vars="unknown_variable_name,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure"
)

Requesting solar dataset from OMNI_HRO2_1MIN...
{
"HAPI": "2.0",
"status": {"code": 1407, "message": "Bad request - unknown dataset parameter"}}




HTTPError: 400 Client Error: 400 for url: https://cdaweb.gsfc.nasa.gov/hapi/data?id=OMNI_HRO2_1MIN&time.min=1500-11-21T00%3A00%3A00Z&time.max=1501-11-22T00%3A00%3A00Z&parameters=unknown_variable_name%2CBX_GSE%2CBY_GSM%2CBZ_GSM%2Cflow_speed%2Cproton_density%2CPressure

#### (Potential edge case) Partial overlap of time range

In [75]:
fetch_data_omni_hro2_1min(
    start_utc="2026-04-13T00:00:00Z",
    end_utc="2026-04-16T00:00:00Z",
    vars="F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure"
)

Requesting solar dataset from OMNI_HRO2_1MIN...
2026-04-13T00:00:00.000Z,4.47,-3.07,-1.13,2.88,464.3,3.54,1.53
2026-04-13T00:01:00.000Z,4.41,-3.08,-1.65,2.64,464.4,3.54,1.53
2026-04-13T00:02:00.000Z,4.47,-3.17,-0.93,2.90,99999.9,999.99,99.99
2026-04-13T00:03:00.000Z,4.49,-2.98,-1.52,2.95,465.4,3.15,1.36
2026-04-13T00:04:00.000Z,4.49,-3.20,-0.85,2.99,465.4,3.15,1.36
2026-04-13T00:05:00.000Z,4.47,-2.88,-1.60,2.98,465.0,3.33,1.44
2026-04-13T00:06:00.000Z,4.44,-3.15,0.38,2.98,465.0,3.33,1.44
2026-04-13T00:07:00.000Z,4.45,-3.20,0.65,3.02,465.0,3.33,1.44
2026-04-13T00:08:00.000Z,4.50,-2.51,2.94,2.22,99999.9,999.99,99.99
2026-04-13T00:09:00.000Z,4.48,-2.25,2.98,2.40,468.4,2.75,1.21
2026-04-13T00:10:00.000Z,4.60,-2.83,-0.47,3.33,468.4,2.75,1.21
2026-04-13T00:11:00.000Z,4.53,-2.97,0.14,3.05,468.4,2.75,1.21
2026-04-13T00:12:00.000Z,4.61,-2.78,-1.48,3.37,99999.9,999.99,99.99
2026-04-13T00:13:00.000Z,4.56,-3.25,-0.34,2.94,464.1,3.31,1.43
2026-04-13T00:14:00.000Z,4.53,-3.52,0.33,2.59,464.1,3.31,1.4

,time,F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure
0,2026-04-13T00:00:00.000Z,4.47,-3.07,-1.13,2.88,464.3,3.54,1.53
1,2026-04-13T00:01:00.000Z,4.41,-3.08,-1.65,2.64,464.4,3.54,1.53
2,2026-04-13T00:02:00.000Z,4.47,-3.17,-0.93,2.90,99999.9,999.99,99.99
3,2026-04-13T00:03:00.000Z,4.49,-2.98,-1.52,2.95,465.4,3.15,1.36
4,2026-04-13T00:04:00.000Z,4.49,-3.20,-0.85,2.99,465.4,3.15,1.36
...,...,...,...,...,...,...,...,...
71,2026-04-13T01:11:00.000Z,9999.99,9999.99,9999.99,9999.99,99999.9,999.99,99.99
72,2026-04-13T01:12:00.000Z,9999.99,9999.99,9999.99,9999.99,99999.9,999.99,99.99
73,2026-04-13T01:13:00.000Z,9999.99,9999.99,9999.99,9999.99,99999.9,999.99,99.99
74,2026-04-13T01:14:00.000Z,4.62,-2.51,-0.50,3.80,99999.9,999.99,99.99


### Return json

#### Success

In [19]:
# success
json_data_success = fetch_data_omni_hro2_1min(
    start_utc="2021-11-21T00:00:00Z",
    end_utc="2021-11-22T00:00:00Z",
    format='json',
    vars="F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure"
)

Requesting solar dataset from OMNI_HRO2_1MIN...
{
"HAPI": "2.0",
"status": {"code": 1200, "message": "OK"},
"format": "json",
"parameters": [
{
"name": "Time",
"type": "isotime",
"units": "UTC",
"length":24,
"fill": null
},
{
"name": "F",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "Magnitude of avg. field vector (nT) (last currently-available OMNI B-field data Apr 12, 2026)"
},
{
"name": "BX_GSE",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "Bx (nT), GSE"
},
{
"name": "BY_GSM",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "By (nT), GSM, determined from post-shift GSE components"
},
{
"name": "BZ_GSM",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "Bz (nT), GSM, determined from post-shift GSE components"
},
{
"name": "flow_speed",
"type": "double",
"units": "km/s",
"fill": "99999.9",
"description": "Flow Speed (km/s), GSE"
},
{
"name": "proton_density",
"type": "double",
"units": "n/cc",
"fill": "9

In [22]:
json_data_success.keys()

dict_keys(['HAPI', 'status', 'format', 'parameters', 'data'])

In [23]:
json_data_success['status']

{'code': 1200, 'message': 'OK'}

#### Fail: no data in time range

In [30]:
# fail
fetch_json_no_data = fetch_data_omni_hro2_1min(
    start_utc="1500-11-21T00:00:00Z",
    end_utc="1501-11-22T00:00:00Z",
    format='json',
    vars="F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure"
)

Requesting solar dataset from OMNI_HRO2_1MIN...
{
"HAPI": "2.0",
"status": {"code": 1200, "message": "OK"},
"format": "json",
"parameters": [
{
"name": "Time",
"type": "isotime",
"units": "UTC",
"length":24,
"fill": null
},
{
"name": "F",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "Magnitude of avg. field vector (nT) (last currently-available OMNI B-field data Apr 12, 2026)"
},
{
"name": "BX_GSE",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "Bx (nT), GSE"
},
{
"name": "BY_GSM",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "By (nT), GSM, determined from post-shift GSE components"
},
{
"name": "BZ_GSM",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "Bz (nT), GSM, determined from post-shift GSE components"
},
{
"name": "flow_speed",
"type": "double",
"units": "km/s",
"fill": "99999.9",
"description": "Flow Speed (km/s), GSE"
},
{
"name": "proton_density",
"type": "double",
"units": "n/cc",
"fill": "9

In [31]:
fetch_json_no_data

'{\n"HAPI": "2.0",\n"status": {"code": 1200, "message": "OK"},\n"format": "json",\n"parameters": [\n{\n"name": "Time",\n"type": "isotime",\n"units": "UTC",\n"length":24,\n"fill": null\n},\n{\n"name": "F",\n"type": "double",\n"units": "nT",\n"fill": "9999.99",\n"description": "Magnitude of avg. field vector (nT) (last currently-available OMNI B-field data Apr 12, 2026)"\n},\n{\n"name": "BX_GSE",\n"type": "double",\n"units": "nT",\n"fill": "9999.99",\n"description": "Bx (nT), GSE"\n},\n{\n"name": "BY_GSM",\n"type": "double",\n"units": "nT",\n"fill": "9999.99",\n"description": "By (nT), GSM, determined from post-shift GSE components"\n},\n{\n"name": "BZ_GSM",\n"type": "double",\n"units": "nT",\n"fill": "9999.99",\n"description": "Bz (nT), GSM, determined from post-shift GSE components"\n},\n{\n"name": "flow_speed",\n"type": "double",\n"units": "km/s",\n"fill": "99999.9",\n"description": "Flow Speed (km/s), GSE"\n},\n{\n"name": "proton_density",\n"type": "double",\n"units": "n/cc",\n"fill"

In [71]:
fetch_json_no_data.replace('\n', '')

'{"HAPI": "2.0","status": {"code": 1200, "message": "OK"},"format": "json","parameters": [{"name": "Time","type": "isotime","units": "UTC","length":24,"fill": null},{"name": "F","type": "double","units": "nT","fill": "9999.99","description": "Magnitude of avg. field vector (nT) (last currently-available OMNI B-field data Apr 12, 2026)"},{"name": "BX_GSE","type": "double","units": "nT","fill": "9999.99","description": "Bx (nT), GSE"},{"name": "BY_GSM","type": "double","units": "nT","fill": "9999.99","description": "By (nT), GSM, determined from post-shift GSE components"},{"name": "BZ_GSM","type": "double","units": "nT","fill": "9999.99","description": "Bz (nT), GSM, determined from post-shift GSE components"},{"name": "flow_speed","type": "double","units": "km/s","fill": "99999.9","description": "Flow Speed (km/s), GSE"},{"name": "proton_density","type": "double","units": "n/cc","fill": "999.99","description": "Proton density (n/cc) (last currently-available OMNI plasma data Apr 12, 20

#### Fail: Incorrect variable name

In [32]:
# fail: incorrect variable name
fetch_json_incorrect_variable_name = fetch_data_omni_hro2_1min(
    start_utc="2021-11-21T00:00:00Z",
    end_utc="2021-11-22T00:00:00Z",
    format='json',
    vars="unknown_variable_name,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure"
)

Requesting solar dataset from OMNI_HRO2_1MIN...
{
"HAPI": "2.0",
"status": {"code": 1407, "message": "Bad request - unknown dataset parameter"}}




HTTPError: 400 Client Error: 400 for url: https://cdaweb.gsfc.nasa.gov/hapi/data?id=OMNI_HRO2_1MIN&time.min=2021-11-21T00%3A00%3A00Z&time.max=2021-11-22T00%3A00%3A00Z&format=json&parameters=unknown_variable_name%2CBX_GSE%2CBY_GSM%2CBZ_GSM%2Cflow_speed%2Cproton_density%2CPressure

#### (Potential edge case) Partial overlap of time range

In [77]:
fetch_json_edge_case = fetch_data_omni_hro2_1min(
    start_utc="2026-04-13T00:00:00Z",
    end_utc="2026-04-16T00:00:00Z",
    vars="F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure",
    format='json'
)

Requesting solar dataset from OMNI_HRO2_1MIN...
{
"HAPI": "2.0",
"status": {"code": 1200, "message": "OK"},
"format": "json",
"parameters": [
{
"name": "Time",
"type": "isotime",
"units": "UTC",
"length":24,
"fill": null
},
{
"name": "F",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "Magnitude of avg. field vector (nT) (last currently-available OMNI B-field data Apr 12, 2026)"
},
{
"name": "BX_GSE",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "Bx (nT), GSE"
},
{
"name": "BY_GSM",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "By (nT), GSM, determined from post-shift GSE components"
},
{
"name": "BZ_GSM",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "Bz (nT), GSM, determined from post-shift GSE components"
},
{
"name": "flow_speed",
"type": "double",
"units": "km/s",
"fill": "99999.9",
"description": "Flow Speed (km/s), GSE"
},
{
"name": "proton_density",
"type": "double",
"units": "n/cc",
"fill": "9

In [83]:
json.loads(fetch_json_edge_case).keys()

dict_keys(['HAPI', 'status', 'format', 'parameters', 'data'])

In [81]:
print(fetch_json_edge_case)

{
"HAPI": "2.0",
"status": {"code": 1200, "message": "OK"},
"format": "json",
"parameters": [
{
"name": "Time",
"type": "isotime",
"units": "UTC",
"length":24,
"fill": null
},
{
"name": "F",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "Magnitude of avg. field vector (nT) (last currently-available OMNI B-field data Apr 12, 2026)"
},
{
"name": "BX_GSE",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "Bx (nT), GSE"
},
{
"name": "BY_GSM",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "By (nT), GSM, determined from post-shift GSE components"
},
{
"name": "BZ_GSM",
"type": "double",
"units": "nT",
"fill": "9999.99",
"description": "Bz (nT), GSM, determined from post-shift GSE components"
},
{
"name": "flow_speed",
"type": "double",
"units": "km/s",
"fill": "99999.9",
"description": "Flow Speed (km/s), GSE"
},
{
"name": "proton_density",
"type": "double",
"units": "n/cc",
"fill": "999.99",
"description": "Proton density (n/cc) (l

# Try modularizing code or at least specify interface

In [ ]:
from src.io.atomic import _atomic_write_json, write_success, write_failed
from typing import Any, Dict, Iterable, Iterator, List, Optional, Tuple
from datetime import datetime, timedelta, timezone
from dataclasses import dataclass
import time

CLI_UTC_FMT = "%Y-%m-%d %H:%M:%S"
HAPI_UTC_FMT = "%Y-%m-%dT%H:%M:%SZ"

def parse_cli_utc_datetime(value: str) -> datetime:
    """Parse CLI UTC string into UTC-naive datetime; reject ISO T/Z strings."""

def parse_hapi_utc_datetime(value: str) -> datetime:
    """Parse /info HAPI timestamp into UTC-naive datetime."""

def format_hapi_utc_datetime(value: datetime) -> str:
    """Format UTC-naive datetime for HAPI /data request."""

def _chunk_token(dt_: Optional[datetime]) -> str:

    """
    Returns the provided in datetime UTC format with non alphanumerics removed.
    
    Main use case is for chunk filenames
    """

    if dt_ is None:
        return "open"
    
    # defensive on the invariant that strdatetimes are UTC naive (no tzinfo)
    if dt_.tzinfo is not None:
        dt_ = dt_.astimezone(timezone.utc).replace(tzinfo=None)

    # Example: 20250101T000000Z
    return dt_.strftime("%Y%m%dT%H%M%SZ")

def _run_id_utc() -> str:
    """
    Returns the current datetime in UTC format with non alphanumerics removed.

    Main use case is an identifier proxy (generating run ids)
    """
    # Example: 20251229T103210Z
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def omni_chunk_filename(chunk_start: datetime, chunk_end: datetime) -> str:
    return f"chunk_{_chunk_token(chunk_start)}__{_chunk_token(chunk_end)}.json"



# @dataclass autogenerates "dunder" methods, frozen=True makes instances immutable
@dataclass(frozen=True)
class OmniChunk:
    chunk_start: datetime
    chunk_end: datetime
    payload: dict


@dataclass(frozen=True) 
class OmniIngestionPlan:
    requested_start: datetime
    requested_end: datetime
    effective_start: datetime
    effective_end: datetime
    time_range_overlap_status: str  # "subset" | "partial" | "full"
    warnings: list[str]
    parameters: list[str]


def write_chunk_json(run_dir: Path, chunk: OmniChunk) -> Path:
    path = run_dir / omni_chunk_filename(chunk.chunk_start, chunk.chunk_end)
    _atomic_write_json(path, chunk.payload)
    return path


def fetch_hapi_info(base_url: str,
                    dataset_id: str,
                    timeout_s: int) -> dict:
    """
    Fetch dataset metadata at runtime from `/info` CDAWeb HAPI endpoint.
    Malformed dataset_id is expected to be caught as an exception.

    Args:
    - base_url: the base URL for the CDAWeb HAPI
    - dataset_id: the dataset id to fetch data from (e.g. "OMNI_HRO2_1MIN")
    - timeout_s: how long to wait before GET request fails
        

    Returns:
        

    Raises:
        
    """
    info_url = base_url + '/info'
    response = requests.get(info_url, params={"id": dataset_id}, timeout=timeout_s)

    try:
        payload = response.json()
    except ValueError as exc:
        response.raise_for_status()
        raise RuntimeError("CDAWeb HAPI /info returned non-JSON response") from exc

    hapi_status = payload.get("status", {})
    hapi_code = hapi_status.get("code")
    hapi_message = hapi_status.get("message")

    if response.status_code >= 400 or hapi_code != 1200:
        raise RuntimeError(
            f"CDAWeb HAPI /info failed for dataset_id={dataset_id} "
            f"| http_status={response.status_code} "
            f"| hapi_status={hapi_code} "
            f"| message={hapi_message}"
        )
    
    return payload

def validate_hapi_info(
    info: dict,
    supported_hapi_version: str,
    requested_parameters: list[str],
    start: datetime,
    end: datetime,
) -> OmniIngestionPlan:
    """
    Validates CLI args (start date, end date, requested parameters)
    against the /info HAPI payload. 

    - accepts start: datetime, end: datetime
    - parses /info startDate/stopDate internally
    - compares datetime intervals
    - returns a small validated request plan

    /info response keys
    ['HAPI', 'resourceURL', 'contact', 'parameters', 'startDate', 'stopDate', 'status']
    """
    
    # HAPI version mismatch -> RuntimeError
    current_hapi_version = info['HAPI']
    if supported_hapi_version != current_hapi_version:
        raise RuntimeError(f"Supported HAPI version ({supported_hapi_version}) does not match current HAPI version ({current_hapi_version})")

    # requested parameter missing -> ValueError
    parameters_glossary = set([param_metadata['name'] for param_metadata in info['parameters']])
    requested_parameters_set = set(requested_parameters)
    if not requested_parameters_set.issubset(parameters_glossary):
        raise ValueError(f"Unsupported parameters found: {requested_parameters_set - parameters_glossary}")

    # time range outside info startDate/stopDate -> ValueError
    # parse /info HAPI startDate and stopDate for date operations
    dataset_start  = parse_hapi_utc_datetime(info['startDate'])
    dataset_stop  = parse_hapi_utc_datetime(info['stopDate'])
    # perform the following
    """
    - If requested end <= dataset startDate, raise ValueError.
    - If requested start >= dataset stopDate, raise ValueError.
    - If intervals partially overlap, warn and continue.
    - If requested interval is fully inside dataset interval, continue silently.
    """
    if end <= dataset_start or start >= dataset_stop:
        raise ValueError(f"Date interval request [{start}, {end}] falls beyond dataset record period [{dataset_start}, {dataset_stop}]")

    # if no ValueError is raised, this means we have end > dataset_start and start < dataset_stop 
    effective_start = max(start, dataset_start)
    effective_end = min(end, dataset_stop)

    time_range_overlap_status = 'subset'
    if effective_start != start or effective_end != end:
        # warn and record partial overlap
        time_range_overlap_status = 'partial'
        pass

    return OmniIngestionPlan(
        requested_start=start, requested_end=end,
        effective_start=effective_start, effective_end=effective_end,
        time_range_overlap_status=time_range_overlap_status,
        parameters=requested_parameters
    )



def fetch_hapi_data(
    base_url: str,
    dataset_id: str,
    parameters: list[str],
    start: datetime,
    end: datetime,
    timeout_s: int,
) -> str:
    """
    Fetch solar wind observations from OMNI via
    `/data` CDAWeb HAPI endpoint.

    Assumptions:
    - parameters, base_url, dataset_id, start, date, are valid

    Args:
        

    Returns:
        

    Raises:
        
    """

    start_utc = format_hapi_utc_datetime(start)
    end_utc = format_hapi_utc_datetime(end)
    params_request = ','.join(parameters)

    data_url = base_url + '/data'

    response = requests.get(data_url,
                            params={"id": dataset_id,
                                    "parameters": params_request,
                                    "time.min": start_utc,
                                    "time.max": end_utc,
                                    "format": "json"},
                                    timeout=timeout_s)
    
    # parse response to json here or return response.text?
    return response.text

def iter_omni_chunks(
    base_url: str,
    dataset_id: str,
    parameters: list[str],
    start: datetime,
    end: datetime,
    timeout_s: int,
    chunk_days: int,
    sleep_s: float
) -> Iterator[OmniChunk]:
    """
    accepts start: datetime, end: datetime only
    performs arithmetic
    calls fetch_hapi_data with datetime chunk boundaries
    """
    chunk_start = start
    while chunk_start < end:
        chunk_end = min(chunk_start + timedelta(days=chunk_days), end)
        payload = fetch_hapi_data(base_url, dataset_id, parameters, chunk_start, chunk_end, timeout_s)
        # possibly return an OmniChunk data class
        # with attributes: payload, chunk_start, chunk_end
        yield OmniChunk(
            chunk_start=chunk_start,
            chunk_end=chunk_end,
            payload=payload,
        )

        # sleep between requests
        time.sleep(float(sleep_s))

        
        chunk_start = chunk_end



def write_manifest(run_dir: Path,
                   status: str,
                   Optional[Dict[str, Any]] = None) -> None:
    """
    Creates or updates a run manifest with a certain status.

    Args:
        

    Returns:
        

    Raises:
        
    """
    pass

def ingest_omni_run(
    omni_config: dict,
    *,
    parameters: list[str],
    start: object,
    end: object,
    raw_base_dir: object | None = None,
) -> Path:

    """
    parse and validate CLI start/end once 
    call `fetch_hapi_info` to fetch /info
    call `validate_hapi_info` using parsed datetimes
    call `iter_omni_chunks` with datetimes
    """
    
    # cache config outputs here first so that any modification 
    # to config keys can be done only here
    CONFIG_DATA_ID = omni_config['hapi']['dataset_id']
    CONFIG_SUPPORTED_HAPI_VER = omni_config['hapi']['supported_version']
    CONFIG_BASE_URL = omni_config['hapi']['base_url']
    CONFIG_CHUNK_DAYS = omni_config['hapi']['chunk_days']
    CONFIG_TIMEOUT = omni_config['hapi']['timeout_s']
    CONFIG_SLEEP = omni_config['hapi']['sleep_s']

    if not raw_base_dir:
        raw_base_dir = omni_config['hapi']['raw_output_dir']

    # parse and validate CLI start/end once
    start_dt = parse_cli_utc_datetime(start)
    end_dt = parse_cli_utc_datetime(end)
    
    # fetch /info for configured dataset_id specified in config
    info = fetch_hapi_info(CONFIG_BASE_URL, CONFIG_DATA_ID, CONFIG_TIMEOUT)

    # validate HAPI version, CLI args
    plan = validate_hapi_info(
        info,
        supported_hapi_version=CONFIG_SUPPORTED_HAPI_VER,
        requested_parameters=parameters,
        start=start_dt,
        end=end_dt,
    )

    # create runid + rundir at <raw_output_dir>/<dataset_id>/run_id=<run_id>
    run_id = _run_id_utc()
    # e.g. "data/01-raw/omni/OMNI_HRO2_1MIN"
    data_dir = Path(raw_base_dir) / CONFIG_DATA_ID
    # e.g. "data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260323T135622Z"
    run_dir = data_dir / f"run_id={run_id}"

    # write RUNNING manifest
    write_manifest(run_dir=run_dir,
                   status='RUNNING')

    # write dataset metadata from /info response 
    _atomic_write_json(data_dir / "hapi_info.json", info)


    try:
        total_rows = 0
        chunk_files = []
        chunk_statuses = []
        
        # iterate /data chunks
        for chunk in iter_omni_chunks(base_url=CONFIG_BASE_URL,
                                      dataset_id=CONFIG_DATA_ID,
                                      parameters=parameters,
                                      start=plan.effective_start,
                                      end=plan.effective_end,
                                      timeout_s=CONFIG_TIMEOUT,
                                      chunk_days=CONFIG_CHUNK_DAYS,
                                      sleep_s=CONFIG_SLEEP):
            
            # write raw chunk JSON files
            out_path = write_chunk_json(run_dir, chunk)

            # record information about the recently retrieved chunk
            rows = len(chunk.payload.get("data", []))
            status = chunk.payload.get("status", {})
            chunk_files.append(out_path.name)
            total_rows += rows
            chunk_statuses.append({
                "file": out_path.name,
                "chunk_start_utc_str": format_hapi_utc_datetime(chunk.chunk_start),
                "chunk_end_utc_str": format_hapi_utc_datetime(chunk.chunk_end),
                "hapi_status_code": status.get("code"),
                "hapi_status_message": status.get("message"),
                "rows": rows,
            })


        # write _SUCCESS
        write_success(run_dir)

        # write SUCCESS manifest
        write_manifest(run_dir=run_dir,
                   status='SUCCESS')
        
    except Exception as e:
        # write _FAILED
        write_failed(run_dir)

        # write FAILED manifest
        write_manifest(run_dir=run_dir,
                   status='SUCCESS')

        # reraise the original exception with full traceback.
        raise

In [87]:
omni_config

{'hapi': {'base_url': 'https://cdaweb.gsfc.nasa.gov/hapi',
  'supported_version': '2.0',
  'dataset_id': 'OMNI_HRO2_1MIN',
  'chunk_days': 10,
  'timeout_s': 120,
  'raw_output_dir': 'data/01-raw/omni'}}

In [89]:
CLI_PARAM = "F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure"
CLI_START = "2021-11-21T00:00:00Z"
CLI_END = "2021-11-22T00:00:00Z"
CONFIG_DATA_ID = omni_config['hapi']['dataset_id']
CONFIG_SUPPORTED_HAPI_VER = omni_config['hapi']['supported_version']
CONFIG_BASE_URL = omni_config['hapi']['base_url']
CONFIG_CHUNK_DAYS = omni_config['hapi']['chunk_days']
CONFIG_TIMEOUT = omni_config['hapi']['timeout_s']
CONFIG_OUTPUT_DIR = omni_config['hapi']['raw_output_dir']


In [91]:
fetch_hapi_info_test = fetch_hapi_info(CONFIG_BASE_URL, CONFIG_DATA_ID, CONFIG_TIMEOUT)
print(fetch_hapi_info_test)
print(fetch_hapi_info_test.keys())

{'HAPI': '2.0', 'resourceURL': 'https://cdaweb.gsfc.nasa.gov/misc/NotesO.html#OMNI_HRO2_1MIN', 'contact': 'J.H. King, N. Papatashvilli @ AdnetSystems, NASA GSFC', 'parameters': [{'name': 'Time', 'length': 24, 'units': 'UTC', 'type': 'isotime', 'fill': None}, {'name': 'IMF', 'description': 'OMNI ID code for the source spacecraft for time-shifted IMF values (see OMNI documentation link for codes)', 'units': None, 'type': 'integer', 'fill': '99'}, {'name': 'PLS', 'description': 'OMNI ID code for the source spacecraft  for time-shifted IP plasma values (see OMNI documentation link for codes)', 'units': None, 'type': 'integer', 'fill': '99'}, {'name': 'IMF_PTS', 'description': 'Number of fine time scale points in IMF averages', 'units': None, 'type': 'integer', 'fill': '999'}, {'name': 'PLS_PTS', 'description': 'Number of fine time scale points in plasma averages', 'units': None, 'type': 'integer', 'fill': '999'}, {'name': 'percent_interp', 'description': 'Percent interpolated', 'units': No

In [92]:
fetch_hapi_info_test

{'HAPI': '2.0',
 'resourceURL': 'https://cdaweb.gsfc.nasa.gov/misc/NotesO.html#OMNI_HRO2_1MIN',
 'contact': 'J.H. King, N. Papatashvilli @ AdnetSystems, NASA GSFC',
 'parameters': [{'name': 'Time',
   'length': 24,
   'units': 'UTC',
   'type': 'isotime',
   'fill': None},
  {'name': 'IMF',
   'description': 'OMNI ID code for the source spacecraft for time-shifted IMF values (see OMNI documentation link for codes)',
   'units': None,
   'type': 'integer',
   'fill': '99'},
  {'name': 'PLS',
   'description': 'OMNI ID code for the source spacecraft  for time-shifted IP plasma values (see OMNI documentation link for codes)',
   'units': None,
   'type': 'integer',
   'fill': '99'},
  {'name': 'IMF_PTS',
   'description': 'Number of fine time scale points in IMF averages',
   'units': None,
   'type': 'integer',
   'fill': '999'},
  {'name': 'PLS_PTS',
   'description': 'Number of fine time scale points in plasma averages',
   'units': None,
   'type': 'integer',
   'fill': '999'},
  {'na

In [100]:
fetch_hapi_info_test['startDate']

'1995-01-01T00:00:00Z'

In [101]:
fetch_hapi_info_test['stopDate']

'2026-04-13T01:15:00Z'